# A2 — Uncertainty from scratch

**From:** "I computed a number on 15 data points"  **To:** putting honest error bars on any statistic with the bootstrap, and knowing exactly what a confidence interval does (and does not) claim.

Yesterday's pipeline measured call `swz_MUL0056`: median response gap **1,720ms** — computed from just **15** handoffs. Is 1,720 the truth about this agent, or did we get a weird 15? That doubt has a science to it.

A number computed from a sample (a mean, a median, a kappa…) is an **estimate**. Different samples → different estimates. The spread of would-be estimates is **sampling error** — and we can SEE it by playing god for a moment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

population = rng.lognormal(mean=6.4, sigma=0.7, size=100_000)     # pretend: ALL gaps this agent will ever produce
true_median = np.median(population)
print(f"true population median (god view): {true_median:.0f}ms")

medians_n15 = [np.median(rng.choice(population, 15)) for _ in range(2000)]
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(medians_n15, bins=50)
ax.axvline(true_median, color="tab:red", lw=2, label="true median")
ax.set_xlabel("median of a 15-gap sample"); ax.set_ylabel("count")
ax.set_title("2000 parallel universes, each measuring 15 gaps"); ax.legend(); plt.show()
print(f"estimates ranged {min(medians_n15):.0f} to {max(medians_n15):.0f}ms - same agent, different luck")

Read it: every bar is a universe where we measured the *same* agent with a *different* 15 handoffs. Some universes concluded 1,300ms; others 2,400ms. Our 1,720 is one draw from this spread.

**PREDICT:** if each universe measured 100 gaps instead of 15, does the spread widen or shrink? By roughly what factor if we go 15 → 60 (4× the data)?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
for n, color in [(15, "tab:blue"), (60, "tab:orange"), (240, "tab:green")]:
    meds = [np.median(rng.choice(population, n)) for _ in range(2000)]
    ax.hist(meds, bins=50, alpha=0.55, color=color, label=f"n={n}  (spread sd={np.std(meds):.0f})")
ax.axvline(true_median, color="tab:red", lw=2)
ax.set_xlabel("sample median (ms)"); ax.set_ylabel("count"); ax.legend()
ax.set_title("more data -> narrower spread (roughly 1/sqrt(n))"); plt.show()

Quadrupling the data roughly *halves* the spread — the famous **1/√n** law. This is why our calibration block insists on 40–60 labels, not 10: below that, the error bars swallow the conclusion.

## The confidence interval, stated honestly
A **95% confidence interval** is a *recipe* for turning a sample into a range, built so that **across many repeated experiments, the range traps the true value ~95% of the time.** Subtle but important: any single interval either contains the truth or it does not — the 95% describes the *recipe's* long-run hit rate, not a probability about one interval. The cleanest way to internalize that is to watch the recipe play out:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
hits = 0
for i in range(100):
    sample = rng.choice(population, 60)
    boots = [np.median(rng.choice(sample, len(sample))) for _ in range(300)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    ok = lo <= true_median <= hi
    hits += ok
    ax.plot([lo, hi], [i, i], color="tab:green" if ok else "tab:red", lw=1.5)
ax.axvline(true_median, color="black", lw=2)
ax.set_xlabel("ms"); ax.set_ylabel("experiment #")
ax.set_title(f"100 intervals from 100 samples - {hits} trapped the truth")
plt.show()

Roughly 95 green, a handful of red — *by design*. The red ones are not mistakes; they are the honest 5%.

## But real life gives you ONE sample — enter the bootstrap
No population, no parallel universes. The bootstrap's move: **let the sample impersonate the population.** Resample your own n points *with replacement* (some points repeat, some sit out), recompute the statistic, repeat thousands of times — the spread of those recomputations approximates the sampling error. It feels like cheating; it is provably reasonable; you already saw it work in the plot above (each interval came from bootstrapping one sample).

Now do it for real — `MUL0056`'s actual 15 gaps:

In [ ]:
import json
from signals import turn_metrics
call = json.loads((ROOT / "data" / "normalized" / "swz_MUL0056.json").read_text())
gaps = np.array([e["gap_ms"] for e in turn_metrics(call["turns"])
                 if e["prev_spk"] == "user" and e["next_spk"] == "agent" and e["fto_ms"] >= 0])
boots = np.array([np.median(rng.choice(gaps, len(gaps))) for _ in range(4000)])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"n={len(gaps)} · point estimate median={np.median(gaps):.0f}ms · 95% CI [{lo:.0f}, {hi:.0f}]ms")
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.hist(boots, bins=40)
ax.axvline(lo, color="tab:red"); ax.axvline(hi, color="tab:red")
ax.set_xlabel("bootstrap median (ms)"); ax.set_ylabel("count")
ax.set_title("bootstrap distribution, real call, n=15"); plt.show()

Read the width of that interval out loud. With n=15, "median 1,720ms" honestly means "somewhere in the high hundreds to mid-two-thousands" — still clearly laggy (the whole interval sits above 800ms!), but a single crisp number would have overclaimed precision. **An estimate without an interval is a vibe.**

This exact machinery returns in Block 7 / book F3: kappa gets a bootstrap CI, and the claim rules read the *interval*, not the point.

## Self-check
1. What is sampling error, in one sentence?
2. The 1/√n law: to halve your error bars, multiply data by …?
3. State precisely what "95%" in a 95% CI refers to.
4. Why does resampling *your own data* tell you anything new?
5. **Gotcha:** a colleague runs 50,000 bootstrap iterations instead of 4,000 "to get a tighter interval." What do you tell them?

<details><summary>Answers</summary>

1. The spread among estimates that different same-size samples from the same truth would produce.
2. ×4.
3. The long-run trap rate of the interval-building recipe across repeated experiments — not a probability statement about any single interval.
4. The sample's internal variability approximates the population's; resampling replays "alternative samples you could have drawn" using the best stand-in you have.
5. Iterations only smooth the *picture* of the spread; the width is driven by n (the real data). Tighter intervals are bought with more data, not more loops.
</details>